# packet_analyze_v2 — Kaggle 올인원 (pcap → 판정)

## ⚠️ 실행 전 (오른쪽 사이드바 'Notebook options')
1. **Accelerator → GPU** (T4 x2 또는 P100)
2. **Internet → On** (전화인증 필요. 안 켜면 git/apt/ollama 전부 실패)
3. **Add Data → Upload** 로 분석할 pcap을 데이터셋으로 올리기 → `/kaggle/input/...`에 마운트됨

흐름: clone → 분석도구 설치(Suricata+Zeek) → Ollama+qwen2.5:14b → pcap 탐색 → analyze.sh → llm_analyze

In [ ]:
# 1. repo clone (Kaggle 작업 디렉토리 = /kaggle/working)
REPO = "https://github.com/DAADAISMYLIFE/packet_analyze_v2.git"
!rm -rf /kaggle/working/packet_analyze_v2
!git clone -q $REPO /kaggle/working/packet_analyze_v2
%cd /kaggle/working/packet_analyze_v2
!git log --oneline -1

In [ ]:
%%bash
# 2. 분석도구 설치: Suricata(+ET Open 룰), Zeek(네이티브), tcpdump
set -e
echo '[1/3] Suricata + tcpdump ...'
apt-get -qq update
apt-get -qq install -y suricata tcpdump >/dev/null
echo '[2/3] ET Open 룰 다운로드 ...'
suricata-update --no-test || echo '⚠️ suricata-update 경고 (계속)'
RULES=/var/lib/suricata/rules/suricata.rules
if [ -s "$RULES" ]; then echo "✅ 룰: $(wc -l < $RULES) 줄"; else echo '❌ 룰 없음'; fi
echo '[3/3] Zeek (네이티브) ...'
UBU=$(lsb_release -rs)
echo "deb http://download.opensuse.org/repositories/security:/zeek/xUbuntu_${UBU}/ /" > /etc/apt/sources.list.d/zeek.list
curl -fsSL "https://download.opensuse.org/repositories/security:zeek/xUbuntu_${UBU}/Release.key" | gpg --dearmor > /etc/apt/trusted.gpg.d/zeek.gpg
apt-get -qq update
apt-get -qq install -y zeek >/dev/null
ln -sf /opt/zeek/bin/zeek /usr/local/bin/zeek
echo '── 설치 확인 ──'; zeek --version; suricata -V

In [ ]:
# 3. Ollama 설치 + 서버 시작 + 모델 다운로드
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time
os.makedirs('/kaggle/working/logs', exist_ok=True)
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/kaggle/working/logs/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
time.sleep(3)
print('모델 다운로드 중... (qwen2.5:14b ~9GB)')
subprocess.run(['ollama', 'pull', 'qwen2.5:14b'], check=True)
print('✅ 완료')

In [ ]:
# 4. Ollama 준비 확인
import urllib.request, time
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        print('✅ Ollama 준비 완료'); break
    except Exception:
        time.sleep(3)
        if i % 5 == 0: print(f'  대기 {i*3}s...')
else:
    print('❌ 타임아웃 — logs/ollama.log 확인')
    !tail -20 /kaggle/working/logs/ollama.log

In [ ]:
# 5. pcap 찾기 (/kaggle/input 에 업로드한 데이터셋에서 자동 탐색)
import glob, os, shutil
cands = (glob.glob('/kaggle/input/**/*.pcap', recursive=True)
         + glob.glob('/kaggle/input/**/*.pcapng', recursive=True)
         + glob.glob('/kaggle/input/**/*.cap', recursive=True))
assert cands, 'pcap 없음 — 오른쪽 Add Data로 pcap 데이터셋을 추가했는지 확인'
src = cands[0]
os.makedirs('pcaps', exist_ok=True)
fn = os.path.basename(src)
shutil.copy(src, f'pcaps/{fn}')
PCAP_PATH = f'pcaps/{fn}'
NAME = os.path.splitext(fn)[0]
print('pcap:', PCAP_PATH, '/ NAME =', NAME)
print('(여러 개면 첫 번째 사용:', len(cands), '개 발견)')

In [ ]:
# 6. 수집 → 압축 (Suricata/Zeek → evidence + 드릴다운 번들)
!bash scripts/analyze.sh {PCAP_PATH}

In [ ]:
# 7. LLM 판정
!python -u scripts/llm_analyze.py {NAME} \
    --base-url http://localhost:11434/v1 \
    --model qwen2.5:14b \
    --temperature 0.0

import json
print('\n=== verdict ===')
print(json.dumps(json.load(open(f'report/{NAME}.verdict.json')), ensure_ascii=False, indent=2))